# **Projeto IA Dália - Análise do Dataset**

## **Informações:**

O CSV selecionado para iniciar as análises foi um DataSet da plataforma Kaggle.

> URL:
>
> https://www.kaggle.com/datasets/csafrit2/maternal-health-risk-data

Esse CSV aborda registros coletados de diferentes hospitais, clínicas comunitárias e serviços de saúde materna por meio de um sistema de monitoramento de risco baseado em IOT.

### **Importando Bibliotecas e Materiais:**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import boxplot as bp
import pingouin as pg
import itertools
from scipy.stats import mannwhitneyu

### **Abrindo Dataset:**

In [ ]:
df = pd.read_csv("../data/raw/Maternal Health Risk Data Set.csv")

df.head(10)

### **Extraindo Informações:**

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

> #### **Relatório**
> Temos 1.014 registros e 7 variáveis:
>
> | Coluna | Tipo | O que parece representar |
> | :--- | :--- | :--- |
> | **Age** | inteiro | idade |
> | **SystolicBP** | inteiro | pressão sistólica |
> | **DiastolicBP** | inteiro | pressão diastólica |
> | **BS** | decimal | glicemia |
> | **BodyTemp** | decimal | temperatura corporal |
> | **HeartRate** | inteiro | frequência cardíaca |
> | **RiskLevel** | texto | nível de risco — **nosso alvo**|
>
> - 0 valores nulos;
> - nenhuma coluna estranha de ID;
> - 6 variáveis numéricas + 1 categórica

In [ ]:
df.describe()

In [ ]:
df["RiskLevel"].value_counts()

> #### **Relatório**
> O `RiskLevel` está equilibrado:
>
> 🟢 Low risk: 406 → 40,0%
>
> 🟡 Mid risk: 336 → 33,1%
>
> 🔴 High risk: 272 → 26,8%

In [ ]:
df.sort_values("HeartRate").head(10)

In [ ]:
df.sort_values("BS", ascending=False).head(10)

In [ ]:
df.nunique()

In [ ]:
df[df["HeartRate"] < 40]

> #### **Relatório**
> O HeartRate = 7
>
> São exatamente dois registros, e eles são idênticos, e eles aparecem nos índices 499 e 908.
>
> Isso tem cara de erro de digitação/erro de coleta no dataset original. Uma frequência cardíaca de 7 bpm seria incompatível com um indivíduo vivo, então não faz sentido alimentar o modelo com isso como se fosse uma observação fisiológica válida.

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated()]

In [ ]:
df.duplicated().value_counts()

> #### **Relatório**
>
> Foram identificados 562 registros duplicados, correspondendo a aproximadamente 55,4% da base. 
>
> A presença de informações duplicadas deverá ser considerada na etapa de limpeza para evitar possível vazamento de informação entre os conjuntos de treinamento e teste.

In [ ]:
df_unique = df.drop_duplicates()

df_unique["RiskLevel"].value_counts()

In [ ]:
df_unique["RiskLevel"].value_counts(normalize=True) * 100

In [ ]:
df.hist(figsize=(12, 8))
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 10))

config = [
    ("Age", "Idade por nível de risco"),
    ("SystolicBP", "Pressão sistólica por nível de risco"),
    ("DiastolicBP", "Pressão diastólica por nível de risco"),
    ("BS", "Glicemia por nível de risco"),
    ("BodyTemp", "Temperatura corporal por nível de risco"),
    ("HeartRate", "Frequência cardíaca por nível de risco")
]

for i, (col, title) in enumerate(config):
    row = i // 3
    col_idx = i % 3
    ax = axes[row, col_idx]
    
    df.boxplot(column=col, by="RiskLevel", ax=ax)
    
    ax.set_title(title)
    ax.set_xlabel("Nível de risco")
    ax.set_ylabel(col)

plt.suptitle("")

plt.tight_layout()
plt.show()


In [ ]:
df_numeric = df.drop(columns=["RiskLevel"])

correlation = df_numeric.corr()

correlation

plt.figure(figsize=(8, 6))
plt.imshow(correlation, cmap="coolwarm", interpolation="nearest")
plt.colorbar()

plt.xticks(range(len(correlation.columns)), correlation.columns, rotation=45)
plt.yticks(range(len(correlation.columns)), correlation.columns)

plt.title("Matriz de correlação")
plt.tight_layout()
plt.show()

> #### **Diagnóstico Final**
>
> **Estrutura**
>
> 1.014 registros × 7 variáveis
>
> **Dados Ausentes**
>
> Nenhum valor nulo.
>
> **Classes Originais**
>
> | Risco | Registros |      % |
> | ----- | --------: | -----: |
> | Low   |       406 | 40,04% |
> | Mid   |       336 | 33,14% |
> | High  |       272 | 26,82% |
> 
> **Duplicadas**
>
> 562 registros duplicados → 55,42% da base.
>
> **Anomalia Encontrada**
>
> 2 registros com `HeartRate = 7`, ambos idênticos e classificados como `low risk`.